# Build training table: trailing rolling team stats per game

Combines `game_logs.csv` (traditional box score) and `team_advanced_stats.csv` (advanced box score) across all seasons in `data/raw/`, and produces one row per game with `HOME_`/`AWAY_` prefixed columns: the trailing `N_GAMES`-game average of every stat *entering* that game (never including the game's own result), plus the actual outcome for labels.

Scope decisions baked into this notebook:
- The rolling window **resets every season** (no carryover from the previous season) and **requires a full window** - a team's first `N_GAMES` games of a season are dropped entirely (for `N_GAMES=10`, the 11th game is the first row produced).
- A game is only included once **both** teams have a full trailing window. Because teams don't all reach their Nth game of a season on the same calendar date (bye days, schedule quirks), a handful of games get dropped where one side qualified before the other - reported below, not silent.
- A tiny number of games (so far: NBA Cup neutral-site semifinals/finals) have no home team at all per the NBA's own records - dropped **before** any rolling computation, so they never leak into a team's trailing window either as an output row or as an input game.
- **Rate/percentage stats are never averaged directly.** Averaging a per-game percentage (e.g. `FT_PCT`) is not the same as the percentage over the combined window - and it breaks outright when a single game has a 0-attempt denominator (a real case in this data: a team with 0 FTA has `FT_PCT = NaN`, which would otherwise poison every rolling window it touches). Every rate stat that can be reconstructed from box-score components is instead derived from the *rolled counts* (e.g. `FT_PCT_AVG10 = FTM_AVG10 / FTA_AVG10`, equivalent to sum(FTM)/sum(FTA) over the window). See the "Derive rate stats" section below for exactly which stats this applies to, and which two (`OREB_PCT`, `DREB_PCT`) can't be reconstructed this way and are left as plain averages.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from data_import import io_utils, settings

In [2]:
# Parameters - change N_GAMES and re-run to produce a different trailing window.
N_GAMES = 10
SEASONS = settings.SEASONS
OUTPUT_PATH = settings.PROCESSED_DATA_DIR / f"training_table_avg{N_GAMES}.csv"

## Load raw data

`IS_HOME` is read directly off `game_logs.csv` rather than re-parsed from `MATCHUP` text - a small number of neutral-site games have `MATCHUP` set to "@" for *both* teams, which the import pipeline already resolves (via `BoxScoreSummaryV2`) into a reliable `IS_HOME` column. A few games have no resolvable home team at all (true neutral-site games); those are dropped **here, before anything else**, so they can't contribute to any team's rolling window.

In [3]:
game_logs = pd.concat(
    [io_utils.load_existing(season, "game_logs").assign(SEASON=season) for season in SEASONS],
    ignore_index=True,
)
game_logs["GAME_DATE"] = pd.to_datetime(game_logs["GAME_DATE"])
game_logs["WIN"] = (game_logs["WL"] == "W").astype(int)
game_logs["IS_HOME"] = game_logs["IS_HOME"].astype(bool)

home_counts = game_logs.groupby(["SEASON", "GAME_ID"])["IS_HOME"].transform("sum")
neutral_game_ids = game_logs.loc[home_counts == 0, "GAME_ID"].unique()
if len(neutral_game_ids):
    print(f"dropping {len(neutral_game_ids)} neutral-site game(s) with no home team: {list(neutral_game_ids)}")
    game_logs = game_logs[~game_logs["GAME_ID"].isin(neutral_game_ids)]

game_logs.shape

dropping 2 neutral-site game(s) with no home team: ['0022501229', '0022501230']


(9836, 32)

## Pull in each game's opponent totals

Two of the rate stats we *can* reconstruct (`REB_PCT`, `DEF_RATING`) need the opponent's own box-score line for that same game (e.g. `REB_PCT = REB / (REB + opponent_REB)`). `game_logs.csv` has both teams' rows already, so this is a self-join on `GAME_ID`, not a new data source.

In [4]:
opp = game_logs[["GAME_ID", "TEAM_ID", "REB", "PTS"]].rename(
    columns={"TEAM_ID": "OPP_TEAM_ID", "REB": "OPP_REB", "PTS": "OPP_PTS"}
)
game_logs = game_logs.merge(opp, on="GAME_ID")
game_logs = game_logs[game_logs["TEAM_ID"] != game_logs["OPP_TEAM_ID"]].drop(columns=["OPP_TEAM_ID"])
game_logs.shape

(9836, 34)

In [5]:
# Every advanced column except these six is dropped here because it's re-derived
# from rolled box-score counts below instead (see "Derive rate stats"); GP/W/L/
# W_PCT/MIN also duplicate what's already in game_logs.
#   - OREB_PCT, DREB_PCT: NOT reconstructable from the box score (validated against
#     real data - a simple OREB/(OREB+opp_DREB) formula is off by ~16% on average,
#     and it's a systematic gap, not rounding noise - NBA's own methodology likely
#     folds in team-rebound tracking that isn't exposed in the box score). Left as
#     plain per-game values, averaged directly - the best available option.
#   - PACE, PIE: no clean box-score decomposition available either; averaged directly
#     (this is also standard practice elsewhere for these two specific metrics).
#   - POSS: not a rate stat (it's a per-game count, like PTS), so plain averaging
#     is already correct - but it's also needed as an input to OFF_RATING/DEF_RATING/
#     TM_TOV_PCT below, so it's kept.
ADVANCED_KEEP_COLS = ["GAME_ID", "TEAM_ID", "OREB_PCT", "DREB_PCT", "PACE", "POSS", "PIE"]

team_advanced = pd.concat(
    [io_utils.load_existing(season, "team_advanced_stats").assign(SEASON=season) for season in SEASONS],
    ignore_index=True,
)
team_advanced = team_advanced[ADVANCED_KEEP_COLS + ["SEASON"]]
team_advanced.shape

(9840, 8)

In [6]:
OWN_COUNT_COLS = [
    "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "OREB", "DREB", "REB",
    "AST", "STL", "BLK", "TOV", "PF", "PTS", "PLUS_MINUS", "WIN",
]
OPP_COUNT_COLS = ["OPP_REB", "OPP_PTS"]  # transient - only used to derive REB_PCT/DEF_RATING below
DIRECT_AVG_COLS = ["OREB_PCT", "DREB_PCT", "PACE", "PIE"]  # not reconstructable - average as-is
COUNT_COLS_TO_ROLL = OWN_COUNT_COLS + OPP_COUNT_COLS + ["POSS"] + DIRECT_AVG_COLS

team_game = game_logs.merge(
    team_advanced,
    on=["SEASON", "GAME_ID", "TEAM_ID"],
    how="inner",
    validate="one_to_one",
)
assert len(team_game) == len(game_logs), "expected exactly one advanced-stats row per game_logs row"
team_game = team_game.sort_values(["SEASON", "TEAM_ID", "GAME_DATE"]).reset_index(drop=True)
team_game.shape

(9836, 39)

## Rolling trailing-window averages

Two separate concerns, computed separately on purpose:
1. **Does a full `N_GAMES`-game history exist this season?** - a row-count question, checked on `GAME_DATE` (never null).
2. **What's the mean of each count stat over that window?** - computed with `min_periods=1` so one genuinely undefined value doesn't null out every other, perfectly valid, stat for that window.

A row is only kept once check (1) passes for every count column that feeds the final features.

In [7]:
AVG_COLS = [f"{col}_AVG{N_GAMES}" for col in COUNT_COLS_TO_ROLL]
group_keys = [team_game["SEASON"], team_game["TEAM_ID"]]

shifted = team_game.groupby(["SEASON", "TEAM_ID"])[COUNT_COLS_TO_ROLL].shift(1)
prior_game_count = (
    team_game.groupby(["SEASON", "TEAM_ID"])["GAME_DATE"].shift(1)
    .groupby(group_keys)
    .rolling(window=N_GAMES, min_periods=1)
    .count()
    .reset_index(level=[0, 1], drop=True)
)
rolled = (
    shifted.groupby(group_keys)
    .rolling(window=N_GAMES, min_periods=1)
    .mean()
    .reset_index(level=[0, 1], drop=True)
)

team_game[AVG_COLS] = rolled
team_game.loc[prior_game_count < N_GAMES, AVG_COLS] = pd.NA

dropped = int((prior_game_count < N_GAMES).sum())
team_game = team_game[prior_game_count >= N_GAMES].reset_index(drop=True)
print(f"dropping {dropped} team-game rows without a full {N_GAMES}-game trailing window")
team_game.shape

dropping 1200 team-game rows without a full 10-game trailing window


(8636, 63)

## Derive rate stats from rolled counts

Validated against the real per-game advanced-stats values (full 2023-24 season) before trusting these:

| stat | formula | mean abs error vs. NBA's own value |
|---|---|---|
| FG_PCT, FG3_PCT, FT_PCT | makes / attempts | < 0.0003 |
| EFG_PCT | (FGM + 0.5*FG3M) / FGA | < 0.0003 |
| TS_PCT | PTS / (2*(FGA + 0.44*FTA)) | < 0.0003 |
| AST_PCT | AST / FGM | < 0.0003 |
| AST_TO | AST / TOV | 0.002 |
| AST_RATIO | 100 * AST / (FGA + 0.44*FTA + AST + TOV) | 0.21 (on a ~18-20 scale) |
| TM_TOV_PCT | TOV / POSS | < 0.0003 |
| OFF_RATING | 100 * PTS / POSS | 0.02 (on a ~110-120 scale) |
| REB_PCT | REB / (REB + opp_REB) | 0.014 |
| DEF_RATING | 100 * opp_PTS / POSS | 0.91 (on a ~110-120 scale) |

Each is applied to the *rolled* counts (e.g. `FTM_AVG10`), which is mathematically the same as summing makes and attempts over the window and dividing - not averaging each game's own percentage. `NET_RATING` is just `OFF_RATING - DEF_RATING` once those two are correct.

In [8]:
def avg_col(name):
    return f"{name}_AVG{N_GAMES}"

team_game[avg_col("FG_PCT")] = team_game[avg_col("FGM")] / team_game[avg_col("FGA")]
team_game[avg_col("FG3_PCT")] = team_game[avg_col("FG3M")] / team_game[avg_col("FG3A")]
team_game[avg_col("FT_PCT")] = team_game[avg_col("FTM")] / team_game[avg_col("FTA")]
team_game[avg_col("EFG_PCT")] = (
    team_game[avg_col("FGM")] + 0.5 * team_game[avg_col("FG3M")]
) / team_game[avg_col("FGA")]
team_game[avg_col("TS_PCT")] = team_game[avg_col("PTS")] / (
    2 * (team_game[avg_col("FGA")] + 0.44 * team_game[avg_col("FTA")])
)
team_game[avg_col("AST_PCT")] = team_game[avg_col("AST")] / team_game[avg_col("FGM")]
team_game[avg_col("AST_TO")] = team_game[avg_col("AST")] / team_game[avg_col("TOV")]
team_game[avg_col("AST_RATIO")] = 100 * team_game[avg_col("AST")] / (
    team_game[avg_col("FGA")] + 0.44 * team_game[avg_col("FTA")]
    + team_game[avg_col("AST")] + team_game[avg_col("TOV")]
)
team_game[avg_col("TM_TOV_PCT")] = team_game[avg_col("TOV")] / team_game[avg_col("POSS")]
team_game[avg_col("OFF_RATING")] = 100 * team_game[avg_col("PTS")] / team_game[avg_col("POSS")]
team_game[avg_col("REB_PCT")] = team_game[avg_col("REB")] / (
    team_game[avg_col("REB")] + team_game[avg_col("OPP_REB")]
)
team_game[avg_col("DEF_RATING")] = 100 * team_game[avg_col("OPP_PTS")] / team_game[avg_col("POSS")]
team_game[avg_col("NET_RATING")] = team_game[avg_col("OFF_RATING")] - team_game[avg_col("DEF_RATING")]

DERIVED_RATE_COLS = [
    avg_col(name) for name in [
        "FG_PCT", "FG3_PCT", "FT_PCT", "EFG_PCT", "TS_PCT", "AST_PCT",
        "AST_TO", "AST_RATIO", "TM_TOV_PCT", "OFF_RATING", "REB_PCT",
        "DEF_RATING", "NET_RATING",
    ]
]

# Final feature set: every rolled column except the transient opponent counts
# (OPP_REB/OPP_PTS - only needed above to derive REB_PCT/DEF_RATING), plus the
# derived rate stats.
OPP_AVG_COLS = [avg_col(c) for c in OPP_COUNT_COLS]
FINAL_AVG_COLS = [c for c in AVG_COLS if c not in OPP_AVG_COLS] + DERIVED_RATE_COLS

## Split into one row per game (HOME_ / AWAY_)

In [9]:
OUTCOME_COLS = ["PTS", "WIN"]
ID_COLS = ["GAME_ID", "SEASON", "GAME_DATE"]

home = team_game[team_game["IS_HOME"]].copy()
away = team_game[~team_game["IS_HOME"]].copy()

home = home.rename(columns={
    "TEAM_ID": "HOME_TEAM_ID", "TEAM_ABBREVIATION": "HOME_TEAM_ABBREVIATION",
    **{c: f"HOME_{c}" for c in FINAL_AVG_COLS + OUTCOME_COLS},
})
away = away.rename(columns={
    "TEAM_ID": "AWAY_TEAM_ID", "TEAM_ABBREVIATION": "AWAY_TEAM_ABBREVIATION",
    **{c: f"AWAY_{c}" for c in FINAL_AVG_COLS + OUTCOME_COLS},
})

home_cols = ID_COLS + ["HOME_TEAM_ID", "HOME_TEAM_ABBREVIATION"] + [f"HOME_{c}" for c in FINAL_AVG_COLS + OUTCOME_COLS]
away_cols = ["GAME_ID", "AWAY_TEAM_ID", "AWAY_TEAM_ABBREVIATION"] + [f"AWAY_{c}" for c in FINAL_AVG_COLS + OUTCOME_COLS]

games = home[home_cols].merge(away[away_cols], on="GAME_ID", how="inner", validate="one_to_one")
games = games.drop(columns=["AWAY_WIN"])  # redundant with HOME_WIN (1 - HOME_WIN)

paired_game_ids = set(home["GAME_ID"]) & set(away["GAME_ID"])
one_sided = (set(home["GAME_ID"]) | set(away["GAME_ID"])) - paired_game_ids
print(f"games kept: {len(games)}")
print(f"one-sided games dropped (only one team had a full {N_GAMES}-game window that night): {len(one_sided)}")

games kept: 4295
one-sided games dropped (only one team had a full 10-game window that night): 46


## Home/away differentials

For every stat in `FINAL_AVG_COLS`, add `DIFF_<stat> = HOME_<stat> - AWAY_<stat>` - a single feature capturing which team has the edge, alongside (not replacing) the individual `HOME_`/`AWAY_` columns. Outcome columns (`PTS`, `WIN`) are excluded here since their diff would just reconstruct the label.

In [10]:
DIFF_COLS = [f"DIFF_{c}" for c in FINAL_AVG_COLS]
for c in FINAL_AVG_COLS:
    games[f"DIFF_{c}"] = games[f"HOME_{c}"] - games[f"AWAY_{c}"]

games.shape

(4295, 115)

In [11]:
ordered_cols = (
    ["GAME_ID", "SEASON", "GAME_DATE", "HOME_TEAM_ID", "HOME_TEAM_ABBREVIATION", "AWAY_TEAM_ID", "AWAY_TEAM_ABBREVIATION"]
    + [f"HOME_{c}" for c in FINAL_AVG_COLS]
    + [f"AWAY_{c}" for c in FINAL_AVG_COLS]
    + DIFF_COLS
    + ["HOME_PTS", "AWAY_PTS", "HOME_WIN"]
)
games = games[ordered_cols].sort_values(["SEASON", "GAME_DATE", "GAME_ID"]).reset_index(drop=True)

settings.PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
games.to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(games)} rows, {len(games.columns)} columns to {OUTPUT_PATH}")

wrote 4295 rows, 115 columns to /Users/jacobgipson/NBA Betting Pipeline/NBA_Betting_Pipeline/data/processed/training_table_avg10.csv


In [12]:
games.columns

Index(['GAME_ID', 'SEASON', 'GAME_DATE', 'HOME_TEAM_ID',
       'HOME_TEAM_ABBREVIATION', 'AWAY_TEAM_ID', 'AWAY_TEAM_ABBREVIATION',
       'HOME_FGM_AVG10', 'HOME_FGA_AVG10', 'HOME_FG3M_AVG10',
       ...
       'DIFF_AST_TO_AVG10', 'DIFF_AST_RATIO_AVG10', 'DIFF_TM_TOV_PCT_AVG10',
       'DIFF_OFF_RATING_AVG10', 'DIFF_REB_PCT_AVG10', 'DIFF_DEF_RATING_AVG10',
       'DIFF_NET_RATING_AVG10', 'HOME_PTS', 'AWAY_PTS', 'HOME_WIN'],
      dtype='object', length=115)

## Sanity checks

In [13]:
print(games.groupby("SEASON").size())
nulls = games.isna().sum()
print("columns with nulls:", nulls[nulls > 0].to_dict())
games.head()

SEASON
2022-23    1073
2023-24    1074
2024-25    1075
2025-26    1073
dtype: int64
columns with nulls: {}


,GAME_ID,SEASON,GAME_DATE,HOME_TEAM_ID,HOME_TEAM_ABBREVIATION,AWAY_TEAM_ID,AWAY_TEAM_ABBREVIATION,HOME_FGM_AVG10,HOME_FGA_AVG10,HOME_FG3M_AVG10,...,DIFF_AST_TO_AVG10,DIFF_AST_RATIO_AVG10,DIFF_TM_TOV_PCT_AVG10,DIFF_OFF_RATING_AVG10,DIFF_REB_PCT_AVG10,DIFF_DEF_RATING_AVG10,DIFF_NET_RATING_AVG10,HOME_PTS,AWAY_PTS,HOME_WIN
0,0022200144,2022-23,2022-11-07,1610612766,CHA,1610612764,WAS,40.8,91.6,11.4,...,-0.021868,0.616803,0.010997,-2.020774,-0.016520,-3.138043,1.117268,100,108,0
1,0022200145,2022-23,2022-11-07,1610612753,ORL,1610612745,HOU,40.1,85.1,9.5,...,0.075176,0.950070,-0.004471,2.924630,0.023257,-1.650133,4.574763,127,134,0
2,0022200151,2022-23,2022-11-07,1610612741,CHI,1610612761,TOR,39.5,86.4,11.0,...,-0.491412,-0.377972,0.028599,-5.503159,-0.017682,1.662546,-7.165705,111,97,1
3,0022200159,2022-23,2022-11-09,1610612766,CHA,1610612757,POR,40.2,92.3,10.6,...,0.416608,1.251330,-0.026497,-6.512851,-0.033344,2.087630,-8.600481,95,105,0
4,0022200160,2022-23,2022-11-09,1610612754,IND,1610612743,DEN,41.2,90.8,15.3,...,-0.059924,-1.005632,0.000419,-0.244228,-0.009865,2.912236,-3.156464,119,122,0
